In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [7]:
# Define a 1D convolution layer without padding
conv = nn.Conv1d(in_channels=2, out_channels=1, kernel_size=2, stride=1, padding=0)

In [8]:
# Manually setting the weights and bias
desired_weights = torch.tensor([[[0.5], [0.5]]]) # Using 0.5 just for demonstration
desired_bias = torch.tensor([0.0])

conv.weight.data = desired_weights
conv.bias.data = desired_bias

In [9]:
# Create an example tensor that is 256 x 2 x 256 by concatenating two 256 x 256 tensors
first_tensor = torch.ones((256, 256))
second_tensor = torch.ones((256, 256)) * 2

In [10]:
first_tensor = first_tensor[:,:236]
second_tensor = second_tensor[:,:236]
input_tensor = torch.stack((first_tensor, second_tensor), dim=1)

In [11]:
# Get the output
output = conv(input_tensor)
print(output)
output = output.squeeze()

tensor([[[1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000]],

        [[1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000]],

        [[1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000]],

        ...,

        [[1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000]],

        [[1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000]],

        [[1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000]]],
       grad_fn=<SqueezeBackward1>)


In [12]:
output.shape

torch.Size([256, 236])

In [13]:
print(output)

tensor([[1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000],
        [1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000],
        [1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000],
        ...,
        [1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000],
        [1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000],
        [1.5000, 1.5000, 1.5000,  ..., 1.5000, 1.5000, 1.5000]],
       grad_fn=<SqueezeBackward0>)


In [20]:
weights = conv.weight
weights

Parameter containing:
tensor([[[0.5000],
         [0.5000]]], requires_grad=True)

In [18]:
l1_reg = torch.norm(weights, p=2)

In [19]:
l1_reg

tensor(0.7071, grad_fn=<NormBackward1>)

In [2]:
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm

from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor


In [3]:
import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

#import a library that allows you to reload a module
from importlib import reload

from eval_utils import load_model

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

#my code 
list_of_idxs = []
list_of_batchs = []

In [4]:
import importlib 
import eval_utils
importlib.reload(eval_utils)
from eval_utils import load_model
model_path = Path("/home/gridsan/tmackey/hydra/singlerun/2023-10-29/no_encode_intensity_concat_comp_concat_neg_mask_v3")
model, test_loader, cfg = load_model(model_path, True)

loader = test_loader

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


  0%|          | 0/3785 [00:00<?, ?it/s]

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


CrystDataModule(self.datasets={'train': {'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy train', 'path': '/home/gridsan/tmackey/cdvae/data/perov_5/train.csv', 'prop': 'heat_ref', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}, 'val': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy val', 'path': '/home/gridsan/tmackey/cdvae/data/perov_5/val.csv', 'prop': 'heat_ref', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}], 'test': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy test', 'path': '/home/gridsan/tmackey/cdvae/data/perov_5/test.csv', 'prop': 'heat_ref', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}]}, self.num_workers={'train': 0, 'val': 0, 'test': 0

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/torch_geometric/deprecation.py:13: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [5]:
model.to('cuda:0')

CDVAE(
  (encoder): DimeNetPlusPlusWrap(
    (rbf): BesselBasisLayer(
      (envelope): Envelope()
    )
    (sbf): SphericalBasisLayer(
      (envelope): Envelope()
    )
    (emb): EmbeddingBlock(
      (emb): Embedding(95, 128)
      (lin_rbf): Linear(in_features=6, out_features=128, bias=True)
      (lin): Linear(in_features=384, out_features=128, bias=True)
    )
    (output_blocks): ModuleList(
      (0): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin_up): Linear(in_features=128, out_features=256, bias=True)
        (lins): ModuleList(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): Linear(in_features=256, out_features=256, bias=True)
          (2): Linear(in_features=256, out_features=256, bias=True)
        )
        (lin): Linear(in_features=256, out_features=256, bias=False)
      )
      (1): OutputPPBlock(
        (lin_rbf): Linear(in_features=6, out_features=128, bias=False)
        (lin

In [21]:
for idx, batch in enumerate(loader):
    print(idx)
    list_of_idxs.append(idx)
    list_of_batchs.append(batch)

idx = list_of_idxs[0]
batch = list_of_batchs[0]

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14


In [22]:
def new_dataloader_batch_processor(batch): 
    batch_reserve = batch
    xrd_int = batch_reserve[1]
    xrd_loc = batch_reserve[2]
    atom_spec = batch_reserve[3]
    batch = batch[0]
    return batch_reserve, xrd_int, xrd_loc, atom_spec, batch

In [23]:
batch_reserve, xrd_int, xrd_loc, atom_spec, batch = new_dataloader_batch_processor(batch)

In [24]:
xrd_int

tensor([[100.0000,  89.2229,  40.5148,  ...,   0.0000,   0.0000,   0.0000],
        [ 18.4477, 100.0000,   0.9011,  ...,   0.0000,   0.0000,   0.0000],
        [ 38.6822, 100.0000,  25.5217,  ...,   0.0000,   0.0000,   0.0000],
        ...,
        [ 63.6974, 100.0000,   1.0961,  ...,   0.0000,   0.0000,   0.0000],
        [ 19.3864, 100.0000,   6.1216,  ...,   0.0000,   0.0000,   0.0000],
        [ 16.7381, 100.0000,  14.0806,  ...,   0.0000,   0.0000,   0.0000]])

In [25]:
xrd_loc

tensor([[21.9119, 31.1830, 38.4384,  ...,  0.0000,  0.0000,  0.0000],
        [20.4207, 29.0363, 35.7604,  ...,  0.0000,  0.0000,  0.0000],
        [20.6636, 29.3857, 36.1957,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [20.7704, 29.5393, 36.3871,  ...,  0.0000,  0.0000,  0.0000],
        [22.7456, 32.3858, 39.9423,  ...,  0.0000,  0.0000,  0.0000],
        [20.5295, 29.1928, 35.9553,  ...,  0.0000,  0.0000,  0.0000]])

In [26]:
xrd_loc = xrd_loc[:,:236]
xrd_int = xrd_int[:,:236]
input_tensor = torch.stack((xrd_loc, xrd_int), dim=1)

In [27]:
output_tensor = conv(input_tensor)

In [28]:
output_tensor.shape

torch.Size([256, 1, 236])

In [29]:
output_tensor

tensor([[[60.9559, 60.2030, 39.4766,  ...,  0.0000,  0.0000,  0.0000]],

        [[19.4342, 64.5182, 18.3307,  ...,  0.0000,  0.0000,  0.0000]],

        [[29.6729, 64.6928, 30.8587,  ...,  0.0000,  0.0000,  0.0000]],

        ...,

        [[42.2339, 64.7696, 18.7416,  ...,  0.0000,  0.0000,  0.0000]],

        [[21.0660, 66.1929, 23.0320,  ...,  0.0000,  0.0000,  0.0000]],

        [[18.6338, 64.5964, 25.0180,  ...,  0.0000,  0.0000,  0.0000]]],
       grad_fn=<SqueezeBackward1>)